# Building an MCP Server with the MCP SDK

Build an MCP server that lets AI assistants interact with GitHub's PR review API.

```
AI Assistant  ──MCP──►  MCP Server  ──HTTP──►  GitHub API
```

In [ ]:
!pip install mcp httpx python-dotenv

## Step 1: Setup

In [ ]:
import os
import httpx
from dotenv import load_dotenv
from mcp.server.fastmcp import FastMCP

load_dotenv("../.env")

GITHUB_API_BASE = "https://api.github.com"
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

print(f"Token loaded: {'Yes' if GITHUB_TOKEN else 'No'}")

# Create MCP server using FastMCP
mcp = FastMCP("github-pr-review")

def get_headers():
    headers = {"Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"}
    if GITHUB_TOKEN:
        headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"
    return headers

async def github_request(method: str, endpoint: str, **kwargs):
    async with httpx.AsyncClient() as client:
        response = await client.request(method, f"{GITHUB_API_BASE}{endpoint}", headers=get_headers(), **kwargs)
        return response.json() if response.status_code < 400 else {"error": response.status_code}

## Step 2: Define the two tools (same as server_http.py)

Use `@mcp.tool()` to register **get_pull_request_files** and **create_pr_review**.

In [ ]:
@mcp.tool()
async def get_pull_request_files(
    owner: str,
    repo: str,
    pull_number: int,
    per_page: int = 30,
    page: int = 1
) -> str:
    """Get the list of files changed in a pull request with their diffs and change statistics."""
    params = {"per_page": per_page, "page": page}
    result = await github_request(
        "GET",
        f"/repos/{owner}/{repo}/pulls/{pull_number}/files",
        params=params
    )
    if isinstance(result, list):
        files = []
        for f in result:
            files.append({
                "filename": f["filename"],
                "status": f["status"],
                "additions": f["additions"],
                "deletions": f["deletions"],
                "changes": f["changes"],
                "patch": f.get("patch", "")[:500] + "..." if len(f.get("patch", "")) > 500 else f.get("patch", "")
            })
        return str(files)
    return str(result)


@mcp.tool()
async def create_pr_review(
    owner: str,
    repo: str,
    pull_number: int,
    body: str = ""
) -> str:
    """Create a review comment on a pull request. Using event COMMENT to add a comment, or APPROVE / REQUEST_CHANGES."""
    if not GITHUB_TOKEN:
        return str({"error": "GITHUB_TOKEN is required for creating PR reviews."})
    payload = {"event": "COMMENT"}
    if body:
        payload["body"] = body
    result = await github_request(
        "POST",
        f"/repos/{owner}/{repo}/pulls/{pull_number}/reviews",
        json=payload
    )
    return str(result)

print("Tools registered: get_pull_request_files, create_pr_review")

## Step 3: Test the tools

Call **get_pull_request_files** (no token needed for public repos). **create_pr_review** requires `GITHUB_TOKEN` and write access.

In [ ]:
# 1. Get pull request files (public repo)
result = await get_pull_request_files(owner="octocat", repo="hello-world", pull_number=1)
print("get_pull_request_files:", result[:500] + "..." if len(result) > 500 else result)

# 2. Add review comment – uncomment and set owner/repo/pull_number to a PR you can write to
# result = await create_pr_review(owner="YOU", repo="REPO", pull_number=1, event="COMMENT", body="Tutorial: MCP SDK test")
# print("create_pr_review:", result)

## Step 4: Run the Server

The server runs via stdio. See `server.py` for the complete implementation.

```python
# server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("github-pr-review")

@mcp.tool()
async def get_repository(owner: str, repo: str) -> str:
    # ... implementation

if __name__ == "__main__":
    mcp.run()  # Runs stdio server
```

**Run:** `python server.py`

## Streamable HTTP transport (for production)

Run the MCP server over HTTP with `server_http.py`. Same two tools: **get_pull_request_files**, **create_pr_review**. Health check at `/health`.

**Run locally:** `python server_http.py`

**Configure Cursor for HTTP transport:**
```json
{
  "mcpServers": {
    "github-pr-review": {
      "url": "https://mcp.example.com/mcp"
    }
  }
}
```

## Configure with Cursor

Add to `.cursor/mcp.json`:

```json
{
  "mcpServers": {
    "github-pr-review": {
      "command": "python3",
      "args": ["/path/to/using-mcp-sdk/server.py"],
      "env": {"GITHUB_TOKEN": "your_token"}
    }
  }
}
```